# Micro-Elise NN Viz

Collect activations for Micro-Elise #2 and compute a stable activity-based layout.

In [ ]:
# cell: setup
from pathlib import Path
import sys

import pandas as pd


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "dqn/src").exists() and (path / "hpo/src").exists() and (path / "nn_viz/src").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = repo_root()
for source_root in ["dqn/src", "hpo/src", "distillation/src", "nn_viz/src"]:
    sys.path.insert(0, str(REPO_ROOT / source_root))

from hpo.environments.solar_system_lander.env import DEFAULT_WORLD_MIX, EnvFactory
from nn_viz import RolloutSpec, collect_activations, compute_activity_layout, load_student_network, plot_network_layout

In [ ]:
# cell: load-micro-elise
CHECKPOINT_PATH = Path(r"G:\\Meine Ablage\\rl_lab\\distillation\\runs\\solar_system_lander_10d_elise_stp_student_64x64_20260721T161456Z\\student_checkpoint.pt")
q_net = load_student_network(CHECKPOINT_PATH)
env_factory = EnvFactory("10d", world_mix=DEFAULT_WORLD_MIX)

In [ ]:
# cell: collect-activations-and-layout; requires: setup, load-micro-elise
rollout_specs = [RolloutSpec(world, seed) for world in DEFAULT_WORLD_MIX for seed in range(5)]
rollouts = collect_activations(q_net, env_factory, rollout_specs)
layout = compute_activity_layout(rollouts, q_net, top_edges_per_target=3, output_edges_per_target=10)

nodes = pd.DataFrame([node.__dict__ for node in layout.nodes])
edges = pd.DataFrame([edge.__dict__ for edge in layout.edges])
summary = pd.DataFrame([
    {
        "frames": rollouts.frame_count,
        "nodes": len(layout.nodes),
        "edges": len(layout.edges),
        "h1_active": int((nodes.query('layer == "h1"').activity > 0).sum()),
        "h2_active": int((nodes.query('layer == "h2"').activity > 0).sum()),
    }
])
summary

In [ ]:
# cell: inspect-layout; requires: collect-activations-and-layout
RESULT_DIR = REPO_ROOT / "nn_viz" / "_local_runs" / "micro_elise_02"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

edges_view = edges.assign(
    source=edges.source_layer + "-" + edges.source_index.astype(str),
    target=edges.target_layer + "-" + edges.target_index.astype(str),
)
edges_view.loc[edges_view.target_layer == "out", "target"] = edges_view.target_index.map({0: "noop", 1: "left", 2: "up", 3: "right"})
output_edges = edges_view.query('target_layer == "out"').sort_values('specificity', ascending=False)[["source", "target", "weight", "relevance", "specificity"]]
h2_view = nodes.query('layer == "h2"').sort_values(['x', 'activity'], ascending=[True, False])[["label", "x", "activity"]]
inactive_view = nodes[nodes.layer.isin(["h1", "h2"]) & (nodes.activity <= 1e-6)][["label", "activity"]]

summary.to_csv(RESULT_DIR / "summary.csv", index=False)
nodes.to_csv(RESULT_DIR / "nodes.csv", index=False)
edges_view.to_csv(RESULT_DIR / "edges.csv", index=False)
output_edges.to_csv(RESULT_DIR / "output_edges.csv", index=False)
h2_view.to_csv(RESULT_DIR / "h2_layout.csv", index=False)
inactive_view.to_csv(RESULT_DIR / "inactive_nodes.csv", index=False)

print(f"saved layout inspection to {RESULT_DIR}")
display(output_edges)
display(h2_view.head(20))
display(inactive_view.head(20))

In [ ]:
# cell: plot-layout; requires: inspect-layout
fig = plot_network_layout(layout, output_path=RESULT_DIR / "layout.png")
fig